# 🧠 **Single Agent Pipeline Project**

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


### 🔍 I**nsight: Pipeline Goals**
* Goal: Move from a rigid, step-by-step sequence to an intentional, flexible pipeline.
* Design: Use text analysis to route user queries to specific tools dynamically.

In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expr: str) -> str:
    try:
        return str(eval(expr))
    except:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    try:
        words = text.split()
        return list(set([w.lower() for w in words if len(w) > 4]))[:5]
    except:
        return []

In [3]:
# 🛠️ BONUS TOOL 3: Text Word Counter

def word_counter(text: str) -> str:
    try:
        return f"{len(text.split())} words"
    except:
        return "Error counting words"

### 🔍 **Insight: Tool Modularization**
* Every tool is isolated as an independent function.
* This keeps the codebase modular, allowing you to add new features or tools without breaking existing code.

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [8]:
import json

# 🤖 AGENT FUNCTION (IMPLEMENTED)
def agent(query: str) -> str:
    def log(msg: str):
        print(f"[Log] {msg}")

    try:
        log(f"Input query: '{query}'")
        clean_q = query.strip().rstrip("?.!").lower()

        # Calculator route
        if "calculate" in clean_q or "solve" in clean_q:
            log("Route -> Calculator")
            trigger = "calculate" if "calculate" in clean_q else "solve"
            expr = query[query.lower().find(trigger) + len(trigger):].strip()

            if not expr:
                return json.dumps({"type": "error", "result": "Missing expression"}, indent=2)

            res = calculator(expr)
            if res == "Error in calculation":
                return json.dumps({"type": "error", "result": res}, indent=2)
            return json.dumps({"type": "calculation", "result": res}, indent=2)

        # Keyword route
        elif "keywords" in clean_q or "extract" in clean_q:
            log("Route -> Keyword Extractor")
            trigger = "keywords" if "keywords" in clean_q else "extract"
            txt = query[query.lower().find(trigger) + len(trigger):].strip()

            if txt.lower().startswith("from "):
                txt = txt[4:].strip()
            if not txt:
                return json.dumps({"type": "error", "result": "Missing text"}, indent=2)

            return json.dumps({"type": "keywords", "result": extract_keywords(txt)}, indent=2)

        # Word counter route
        elif "count" in clean_q or "length" in clean_q:
            log("Route -> Word Counter")
            trigger = "count" if "count" in clean_q else "length"
            txt = query[query.lower().find(trigger) + len(trigger):].strip()

            if txt.lower().startswith("words in "): txt = txt[9:].strip()
            if txt.lower().startswith("of "): txt = txt[3:].strip()
            if not txt:
                return json.dumps({"type": "error", "result": "Missing text"}, indent=2)

            return json.dumps({"type": "word_count", "result": word_counter(txt)}, indent=2)

        # Fallback route
        else:
            log("Route -> Fallback Handler")
            return json.dumps({"type": "general", "result": f"Processed: {query}"}, indent=2)

    except Exception as e:
        log(f"Pipeline Error: {e}")
        return json.dumps({"type": "error", "result": str(e)}, indent=2)

### 🔍 **Insight: Agent Routing & Safety**
* The core agent standardizes user inputs and uses clean string matching to direct traffic.
* A main try-except block ensures that formatting errors or invalid operations are caught safely without crashing the system.

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [5]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Count the words in Deep Learning is a subset of AI",
    "What is machine learning?"
]

for q in queries:
    print(f"--- Processing ---")
    print(agent(q))
    print("-" * 30)

--- Processing ---
[Log] Input query: 'Calculate 20 + 5'
[Log] Route -> Calculator
{
  "type": "calculation",
  "result": "25"
}
------------------------------
--- Processing ---
[Log] Input query: 'Extract keywords from Artificial Intelligence is transforming industries'
[Log] Route -> Keyword Extractor
{
  "type": "keywords",
  "result": [
    "industries",
    "artificial",
    "intelligence",
    "transforming"
  ]
}
------------------------------
--- Processing ---
[Log] Input query: 'Count the words in Deep Learning is a subset of AI'
[Log] Route -> Word Counter
{
  "type": "word_count",
  "result": "10 words"
}
------------------------------
--- Processing ---
[Log] Input query: 'What is machine learning?'
[Log] Route -> Fallback Handler
{
  "type": "general",
  "result": "Processed: What is machine learning?"
}
------------------------------


### 🔍 **Observation: Automated Test Results**
* The automated checks confirm that all primary routing paths function correctly.
* Every response outputs as a cleanly structured JSON string using standard double quotes.

In [7]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): 
[Log] Input query: ''
[Log] Route -> Fallback Handler
Response: {
  "type": "general",
  "result": "Processed: "
}
Enter query (type 'exit' to stop): Calculate 20+25*10
[Log] Input query: 'Calculate 20+25*10'
[Log] Route -> Calculator
Response: {
  "type": "calculation",
  "result": "270"
}
Enter query (type 'exit' to stop): ThisIsTesting
[Log] Input query: 'ThisIsTesting'
[Log] Route -> Fallback Handler
Response: {
  "type": "general",
  "result": "Processed: ThisIsTesting"
}
Enter query (type 'exit' to stop): exit


### 🔍 **Observation: Runtime Flexibility**
* Interactive testing shows the agent successfully extracts parameters from conversational phrases.
* The system isolates arguments correctly, filters out unnecessary words, and handles fallback inputs smoothly.

## **Final Conclusion**
* **Dynamic Routing:** The pipeline replaces rigid linear steps with flexible, intent-based conditional routing.
* **Modular Nodes:** Isolating tasks into individual tools (Math, Keywords, Word Counter) keeps the architecture clean and modular.
* **Structured Output:** Every operational path outputs a strictly formatted JSON string to ensure clean, consistent data delivery.
* **Pipeline Visibility:** Step-by-step logs offer transparent tracking of routing decisions during execution.
* **System Resilience:** Comprehensive try-except safety nets and fallback states handle errors safely without dropping or crashing the pipeline.